# Week 1 — Titanic data inspection

This notebook loads a local Titanic `train.csv` and creates descriptive plots. It does not train or report a predictive model.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


In [ ]:
DATA_PATH = Path(
    os.environ.get("TITANIC_TRAIN_CSV", "data/titanic/train.csv")
).expanduser()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Titanic training data was not found at {DATA_PATH}. "
        "Set TITANIC_TRAIN_CSV to the absolute path of train.csv."
    )

df = pd.read_csv(DATA_PATH)
required_columns = {"Survived", "Age", "Embarked", "Pclass"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {sorted(missing_columns)}. "
        f"Found: {sorted(df.columns.tolist())}"
    )
if df.empty:
    raise ValueError(f"Dataset at {DATA_PATH} is empty.")


In [ ]:
# These transformations are for descriptive analysis only.
age_median = df["Age"].median()
if pd.isna(age_median):
    raise ValueError("Age has no non-missing values; cannot compute a median.")
df["Age"] = df["Age"].fillna(age_median)

embarked_mode = df["Embarked"].mode(dropna=True)
if embarked_mode.empty:
    raise ValueError("Embarked has no non-missing values; cannot compute a mode.")
df["Embarked"] = df["Embarked"].fillna(embarked_mode.iloc[0])
df = df.drop(columns=["Cabin"], errors="ignore").drop_duplicates().reset_index(drop=True)
print(f"Loaded {len(df):,} rows from {DATA_PATH}")


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="Age", bins=30)
plt.title("Age distribution")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x="Pclass", hue="Survived")
plt.title("Survival count by passenger class")
plt.tight_layout()
plt.show()
